# Mask pool — aerial RGB boxes through a promptable teacher

One runtime, one **Run all**: download box-annotated aerial data, prompt a
promptable-segmentation teacher with every box, gate what comes back, and
leave a **mask pool** on your Drive — `(image, box prompt, mask)` supervision
in the exact store the rest of this repo already reads.

Builds the **RGB pool**: aerial detection boxes (VisDrone, and optionally DroneVehicle's RGB half) become per-instance teacher masks, gated and stored as `(image, box prompt, mask)` triples for stage B. This is the easy branch — the teachers were trained on RGB — and it exists beside `14_thermal_mask_pool.ipynb`, which is the one this project is actually about. Run this one to grow the RGB side of the encoder's diet and to rehearse the pipeline where the teacher is strongest.

| | |
|---|---|
| **in** | nothing staged by hand — the download cell fetches everything |
| **out** | per-image run-length mask stores + acceptance/calibration reports, zipped to `edgetam-pool/rgb` on your Drive |
| **needs** | a CUDA GPU and ~6 GB of free disk (VisDrone ~1.6 GB, Kust4K's RGB half ~1.7 GB for calibration; +14 GB if `INCLUDE_DRONEVEHICLE`) |
| **takes** | ~10 min to download, ~15 min to calibrate, and roughly 1.5–3 h to harvest VisDrone-train at the default caps (teacher-bound; scale `FRAME_LIMIT` down for a first pass) |

**This notebook trains nothing.** It manufactures and *measures* supervision:
stage B (`07`–`11`) is where training numbers come from, stage C is where the
masklets go. Keeping production separate from consumption is what lets a pool
be inspected, calibrated and staged once, then reused by every run after it.

---

## The pipeline, in one diagram

```
  box dataset          per box:                       teacher            gates
  ┌───────────┐        ┌────────────────┐        ┌──────────────┐   ┌──────────────┐
  │ image     │        │ zoom crop      │        │ SAM 3 (or    │   │ teacher_iou  │
  │  + boxes  │ ─────▶ │ around the box │ ─────▶ │ SAM 2.1),    │──▶│ box_iou      │──▶ RLE store
  │  + class  │        │ (x4 long side, │        │ box-prompted │   │ area         │    per image
  └───────────┘        │  ≥128 px)      │        └──────────────┘   │ component    │  + record.json
                       └────────────────┘                           └──────────────┘
```

Three decisions carried over from the Anti-UAV410 labeller, because they were
measured there:

* **Zoom.** A 15-pixel vehicle in a full frame is a problem teachers fail;
  the same vehicle on a 128-pixel crop resized up is one they solve. The crop
  is the whole trick.
* **Gates.** A teacher mask is kept only if four independent checks agree —
  its own confidence, agreement with the prompting box, area sanity, and
  single-component-ness. What was rejected and *why* is reported per gate;
  the acceptance rate is a measured number, not a hope.
* **The store.** Accepted masks land in the same run-length `.npz` the
  Anti-UAV410 pseudo-labels use, keyed by box index, with a `record.json`
  naming each box, class and verdict. Whatever reads pseudo-masks today reads
  this pool tomorrow; a rejected box is *absent*, which readers already treat
  as "no mask supervision here", not as an empty mask.

## Which teacher, and why it is a setting

`TEACHER` defaults to **`facebook/sam3`** — the strongest promptable
segmenter with a real `transformers` integration (≥5.0): box prompts, batched
crops, SAM 2's API. It is **gated**: open its model page once, accept the
terms, and log in below. Two facts to hold while choosing:

* **`facebook/sam2.1-hiera-large`** is one string away: ungated, Apache-2.0,
  needs only transformers 4.56. If SAM 3's gate or licence is in the way,
  switch and re-run — every report records which teacher made which mask.
* **`facebook/sam3.1` is not an option**, checked rather than assumed: it
  ships as a bare checkpoint (no transformers classes), pins `numpy<2`
  through its GitHub package, and its headline Object Multiplex is a
  many-objects-per-frame throughput win — this labeller prompts one box per
  crop and would gain nothing.

The choice is *measured* anyway: cell 14 scores the configured
teacher against real drawn masks before the harvest spends an hour on it.


In [ ]:
# --- Runtime, repo, GPU -------------------------------------------------
import os, sys
from pathlib import Path

REPO   = Path("/content/sam-dedection")
BRANCH = "claude/encoder-architecture-colab-myo61y"

if not REPO.exists():
    !git clone -q -b {BRANCH} https://github.com/yigitkayabagci/sam-dedection.git {REPO}
!git -C {REPO} fetch -q origin {BRANCH}
!git -C {REPO} checkout -q {BRANCH} && git -C {REPO} merge -q --ff-only origin/{BRANCH}
os.chdir(REPO)
sys.path.insert(0, str(REPO))

# Drop this repo's modules so the fast-forward above actually takes effect:
# `git pull` changes files on disk, not modules Python already imported.
for _stale in [n for n in list(sys.modules) if n.split(".")[0] in ("src", "tools")]:
    del sys.modules[_stale]

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!df -h /content | tail -1

# --- Is this notebook the one the repo expects? -------------------------
# The repo just fast-forwarded itself; the .ipynb is a file you uploaded, and
# the two drift apart silently. This says so at cell 1 instead of forty
# minutes in.
import json as _json
NOTEBOOK = "13_rgb_mask_pool.ipynb"
STAMP    = "d6e99551d0"
_stamps  = REPO / "notebooks" / ".stamps.json"
_want    = _json.loads(_stamps.read_text()).get(NOTEBOOK) if _stamps.is_file() else None
if _want and _want != STAMP:
    print("\n" + "=" * 74)
    print(f"!!  STALE NOTEBOOK -- this file is build {STAMP}, the repo ships {_want}")
    print( "!!  Every cell below is the old version. Re-download and re-upload:")
    print(f"!!    https://github.com/yigitkayabagci/sam-dedection/raw/{BRANCH}/notebooks/{NOTEBOOK}")
    print("=" * 74 + "\n")
else:
    print(f"notebook build {STAMP} matches the repo")


In [ ]:
# --- Does this torch actually have kernels for this GPU? ----------------
# Asked by launching one, not by reading an arch list: on a card newer than
# the torch build everything imports and the first real matmul dies mid-run.
import torch

print(f"torch {torch.__version__}, CUDA {torch.version.cuda}")
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    print(f"{torch.cuda.get_device_name(0)}  sm_{major}{minor}  "
          f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB")
    try:
        probe = torch.randn(256, 256, device="cuda")
        (probe @ probe).sum().item()
        with torch.autocast("cuda", dtype=torch.bfloat16):
            (probe @ probe).sum().item()
        print("a real matmul ran, in float32 and bfloat16 -- this GPU is usable")
    except RuntimeError as exc:
        raise SystemExit(
            f"torch {torch.__version__} cannot run on sm_{major}{minor}: {exc}\n"
            f"In Colab, restore the preinstalled torch rather than letting "
            f"anything downgrade it, then Runtime > Restart session.")
else:
    print("!! no CUDA device -- the teacher will not run")


In [ ]:
# --- Dependencies -------------------------------------------------------
# No EdgeTAM here -- this notebook trains nothing, so the sam2-fork/teacher
# collision the training notebooks manage does not exist. The teacher runs
# through transformers alone. 5.0 is the floor that has the Sam3Tracker
# classes; SAM 2.1 needs only 4.56, so one install covers both choices.
before = torch.__version__
!pip install -q "transformers>=5" hf_transfer tqdm

# The one thing an install here must not do: replace the preinstalled torch.
# Read the version off *disk* (reloading torch always raises); pip cannot
# change the torch already live in this kernel, only a restart can -- which
# is exactly the window this check exists to catch.
import importlib.metadata as _md
installed = _md.version("torch")
assert installed == before, (
    f"pip replaced torch {before} with {installed} on disk. Restore it "
    f"before restarting:  !pip install -q --force-reinstall torch=={before}")

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import transformers
print(f"transformers {transformers.__version__}")

# The contracts everything below depends on, tested with no GPU and no
# dataset. If these fail, nothing after this point is worth running.
!python -m unittest -q tests.test_mask_pool tests.test_masklets 2>&1 | tail -3


In [ ]:
# --- Where things live, and every knob ----------------------------------
DATA_ROOT = Path("/content/data")      # datasets; local disk, never Drive
WORK      = Path("/content/work")      # scratch
POOL      = WORK / "pool"              # the product: stores + records

# The teacher. facebook/sam3 is gated -- accept its terms once, then log in
# below. The ungated fallback needs no account and only transformers>=4.56:
#     TEACHER = "facebook/sam2.1-hiera-large"
TEACHER = "facebook/sam3"
DEVICE  = "cuda"
DTYPE   = "bfloat16"

# The gates, exactly labels.Gates -- see that docstring for why teacher_iou
# is a weak gate at small sizes and box_iou is the load-bearing one.
from src.training.labels import Gates
GATES = Gates()                        # Gates(box_iou=0.6, area=(0.15, 1.3), ...)

ZOOM      = 4.0                        # crop side, in box long-sides
MIN_SIZE  = 128                        # crop floor, px -- context for tiny targets
BATCH     = 8                          # crops per teacher forward
MAX_BOXES = 64                         # per image, largest first (VisDrone
                                       # frames can carry hundreds)
FRAME_LIMIT = None                     # images per dataset; 200 for a smoke run
RESUME     = True                      # skip images whose store exists
SEED       = 0

# Calibration budget: frames sampled (seeded), instances per frame.
CAL_FRAMES    = 150
CAL_PER_FRAME = 6

INCLUDE_DRONEVEHICLE = False           # 13 only: +14 GB, same pipeline
HARVEST_PROMPT = "self"                # 14 only: set from the calibration
                                       # table -- "self" = thermal-direct,
                                       # "pair" = ride the registration
# RGBT234 masklet pass (14 only): sequences and frames are capped for a
# first pass; raise them once the acceptance and the overlays look right.
RGBT234_SEQUENCES  = 24
RGBT234_MAX_FRAMES = 600               # per sequence
RGBT234_CHUNK      = 200               # frames per re-prompt; drift bound

for d in (DATA_ROOT, WORK, POOL):
    d.mkdir(parents=True, exist_ok=True)

def tqdm_over(stream, total, desc):
    from tqdm.auto import tqdm
    return tqdm(stream, total=total, desc=desc)

print(f"teacher {TEACHER}  gates {GATES}")
print(f"pool -> {POOL}")


In [ ]:
# --- Drive: where finished work survives a dead runtime -----------------
from google.colab import drive as _drive
_drive.mount("/content/drive")

MIRROR = Path("/content/drive/MyDrive/edgetam-pool/rgb")
MIRROR.mkdir(parents=True, exist_ok=True)

# Zip one dataset's stores to Drive the moment its harvest finishes. The
# pool is thousands of small files; as a folder copy it would trickle for
# minutes and litter Drive's change log, as one stored (not deflated -- the
# payload is already-compressed npz) zip it lands in seconds and later
# runtimes can pull it back with one copy.
from tools.fetch_datasets import archive_to

def stage_pool(dataset):
    source = POOL / dataset
    if not any(source.rglob("*.npz")):
        print(f"   !! nothing under {source} -- not staging")
        return
    archive_to(source, f"pool_{dataset}", MIRROR)

print(f"staging to {MIRROR}")


In [ ]:
# --- Hugging Face: the teacher's gate -----------------------------------
# facebook/sam3 is a gated repository: open
#     https://huggingface.co/facebook/sam3
# once, accept the terms, then run this cell and paste a token (read scope).
# Skippable entirely when TEACHER is the ungated SAM 2.1.
if "sam3" in TEACHER.lower():
    from huggingface_hub import login, whoami
    try:
        print(f"already logged in as {whoami()['name']}")
    except Exception:
        login()
else:
    print(f"{TEACHER} is ungated -- no login needed")


## Downloading the data

`FETCH` in the settings cell names the datasets; the cell below downloads,
verifies and extracts them into the layout the readers glob. Every URL lives
in `tools/fetch_datasets.py` and was checked against the live host, because
several are served in a way that defeats the obvious approach:

* **VisDrone** rides `snapshot_download` against a plain-file Hub mirror
  (`banu4prasad/VisDrone-Dataset`) — the official release lives behind
  per-file Google Drive links that quota out under Colab's shared egress
  addresses. Third-party mirror, so cell 9 counts what arrived and
  checks the class table against the download before anything is labelled.
* **Kust4K** comes from figshare's per-file endpoints with the publisher's
  md5 — the whole-article link answers HTTP 202 with an empty body (it
  *starts building* the zip and returns before it exists). Only the RGB half
  and the labels are fetched here; the thermal half belongs to notebook 14.
* **DroneVehicle** (optional, `INCLUDE_DRONEVEHICLE`) is the same 28 442-pair
  archive notebooks 09/14 use; its RGB half carries the same oriented boxes.

Downloads resume where the host allows it, archives are deleted once
extracted, and a copy staged in `MyDrive/datasets/` is used **before** the
network is touched — the standing escape hatch for anything quota-shaped.


In [ ]:
# --- Download everything, once ------------------------------------------
from tools.fetch_datasets import fetch

FETCH = [
    # VisDrone2019-DET, YOLO-converted mirror: plain files, snapshot-fetched.
    ("visdrone", DATA_ROOT / "VisDrone", []),
    # Kust4K's RGB half + labels -- not harvested, *calibrated on*: real
    # drawn masks under aerial RGB, the closest thing this branch has to
    # ground truth for the teacher itself.
    ("kust4k",   DATA_ROOT / "Kust4K",   ["rgb", "labels"]),
]
if INCLUDE_DRONEVEHICLE:
    FETCH.append(("dronevehicle", DATA_ROOT / "DroneVehicle", ["train"]))

for name, dest, parts in FETCH:
    marker = Path(dest) / ".fetched"
    if marker.is_file():
        print(f"{name}: already fetched -> {dest}")
        continue
    fetch(name, Path(dest), parts=tuple(parts) or None)
    marker.touch()

!df -h /content | tail -1


In [ ]:
# --- What arrived, and does the reader agree? ---------------------------
from src.training import boxes as B
from src.training.aerial import describe_layout

FRAMES = {}

visdrone_split = sorted((DATA_ROOT / "VisDrone").rglob("*DET-train"))
if not visdrone_split:
    print(describe_layout(DATA_ROOT / "VisDrone"))
    raise SystemExit("no *DET-train under the VisDrone root -- the snapshot "
                     "did not land; re-run the download cell and read its output")
FRAMES["visdrone"] = B.yolo_frames(visdrone_split[0])

if INCLUDE_DRONEVEHICLE:
    FRAMES["dronevehicle_rgb"] = B.dronevehicle_frames(
        DATA_ROOT / "DroneVehicle", modality="rgb")

for name, frames in FRAMES.items():
    print(B.summarise_frames(frames, name)); print()

# The names table is an assumption about a third-party conversion; the
# histogram is the download's own answer. Ten classes, vehicles and people
# dominant -- anything else here means the yaml in the mirror changed and
# `B.VISDRONE_NAMES` must be fixed before a single mask is made.
histogram = B.class_histogram(FRAMES["visdrone"])
unknown = [name for name in histogram if name.startswith("class ")]
assert not unknown, (
    f"label indexes outside VISDRONE_NAMES: {unknown} -- the mirror's class "
    f"table moved; fix src/training/boxes.py before labelling")
print(f"{len(histogram)} classes, all inside VISDRONE_NAMES")


In [ ]:
# --- The teacher, loaded once -------------------------------------------
# build_image_teacher dispatches on the id ("sam3" -> Sam3TrackerModel,
# otherwise Sam2Model) and turns a gated-repo refusal into instructions
# rather than a stack trace.
from src.training.labels import build_image_teacher

teacher = build_image_teacher(TEACHER, device=DEVICE, dtype=DTYPE)
print(f"{teacher.model_id} on {DEVICE} ({DTYPE})")


## Look before spending

A histogram cannot tell you a mask is wrong. Eight frames, every box drawn:
**green** masks passed the gates, **red** boxes are what the teacher failed
and *why*. If the green here does not look like objects, stop — the settings
cell is where the zoom and the gates live, and GPU-hours spent past a bad
panel are not recovered.


In [ ]:
# --- Eight frames, every verdict visible --------------------------------
import numpy as np
import matplotlib.pyplot as plt
from src.training.pool import label_boxes, _read_rgb

_name, _frames = next(iter(FRAMES.items()))
figure, axes = plt.subplots(2, 4, figsize=(22, 10))
for axis, frame in zip(axes.ravel(), _frames[:8]):
    pixels = _read_rgb(frame.image)
    resolved, keep = frame.resolved(pixels.shape[:2])
    if frame.inset:
        pixels = pixels[frame.inset:-frame.inset, frame.inset:-frame.inset]
    chosen = [i for i in range(len(resolved)) if keep[i]][:MAX_BOXES]
    masks, rows = label_boxes(pixels, resolved[chosen], teacher, gates=GATES,
                              zoom=ZOOM, min_size=MIN_SIZE, batch_size=BATCH)
    canvas = pixels.copy()
    for local, mask in masks.items():
        canvas[mask] = 0.45 * canvas[mask] + np.array([0, 140, 0])
    axis.imshow(canvas)
    for row in rows:
        x0, y0, x1, y1 = resolved[chosen[row["i"]]]
        ok = row["verdict"] is None
        axis.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                                     edgecolor="lime" if ok else "red",
                                     linewidth=1.2))
        if not ok:
            axis.text(x0, max(y0 - 3, 0), row["verdict"], color="red", fontsize=7)
    accepted = sum(1 for r in rows if r["verdict"] is None)
    axis.set_title(f"{frame.key}: {accepted}/{len(rows)} accepted", fontsize=9)
    axis.axis("off")
plt.suptitle(f"{_name} through {teacher.model_id}", y=1.0)
plt.tight_layout(); plt.show()


## Calibrate before you spend

VisDrone has boxes and no masks, so nothing in the harvest can say how *good*
a teacher mask is — the gates only say where it sits. Kust4K can: its RGB half
is aerial urban imagery with real semantic masks, and `decompose` turns those
into instances whose boxes prompt the teacher exactly the way the harvest
will. IoU against the drawn mask, per class and per size bucket, is the
number to read before spending teacher-hours — and the small-bucket rows are
the ones that matter, because that is where this project lives.


In [ ]:
# --- Teacher vs drawn masks, on Kust4K's RGB half -----------------------
from src.training.aerial import SPECS
from src.training.pool import calibrate_spec, calibration_table

CALIBRATION = {"rgb": calibrate_spec(
    DATA_ROOT / "Kust4K", SPECS["kust4k"], teacher,
    modality="rgb", prompt="self", limit_frames=CAL_FRAMES,
    per_frame=CAL_PER_FRAME, zoom=ZOOM, min_size=MIN_SIZE,
    batch_size=BATCH, seed=SEED, progress=tqdm_over)}

print(f"{len(CALIBRATION['rgb'])} instances scored\n")
print(calibration_table(CALIBRATION))


## The harvest

Resumable by construction — a frame is *done* when its store exists, so a
dead runtime costs the frame it was on and nothing else — and staged to
Drive **per dataset, as each finishes**, because a session that dies three
datasets in should keep three datasets. Scale `FRAME_LIMIT` down for a first
pass; the caps, the gates and this cell's reports all land in the record
files, so two pools are always comparable.


In [ ]:
from src.training.pool import label_pool, write_index

# --- The harvest: every source, resumable, staged as it finishes --------
REPORTS = {}
for name, frames in FRAMES.items():
    REPORTS[name] = label_pool(
        frames, teacher, POOL, dataset=name, prompt="self", gates=GATES,
        zoom=ZOOM, min_size=MIN_SIZE, batch_size=BATCH, limit=FRAME_LIMIT,
        max_boxes=MAX_BOXES, resume=RESUME, progress=tqdm_over)
    write_index(POOL)
    report = REPORTS[name]
    print(f"{name}: {report['accepted']}/{report['attempted']} accepted "
          f"({report['acceptance_rate']:.1%}), "
          f"{report['resumed']} image(s) resumed")
    stage_pool(name)          # to Drive the moment it exists, not at the end


## What this run produced


In [ ]:
# --- The pool, re-read off disk -----------------------------------------
# Off disk and not from this session's variables, so the statement is true
# across resumed runs and partial sessions alike.
from src.training.pool import pool_report, summarise_pool

REPORT = pool_report(POOL)
print(summarise_pool(REPORT))

total = sum(entry["accepted"] for entry in REPORT.values())
teachers = sorted({t for entry in REPORT.values() for t in entry["teachers"]})
print(f"\n{total} accepted masks across {len(REPORT)} dataset(s), "
      f"teacher(s): {', '.join(teachers)}")
print(f"stores under {POOL}, staged zips under {MIRROR}")

import json as _json2
(WORK / "pool_report.json").write_text(_json2.dumps(REPORT, indent=2))
import shutil as _shutil
_shutil.copy(WORK / "pool_report.json", MIRROR / "pool_report.json")
print("pool_report.json -> Drive")


## What to do with the pool

The stores are `labels.py`'s own format — `pseudo_masks.npz` keyed by box
index, `record.json` beside each saying which box, which class, which verdict
— so stage B reads them the way it already reads Anti-UAV410's pseudo-labels.
Wiring the pool into `07`'s `DATASETS` list is a follow-up change to
`src/training/datasets.py`, deliberately not part of this notebook: this one
produces supervision and *measures* it, it does not train.

## The contamination rule this pool creates

**AeroVIS is VisDrone re-labelled** (plus UAVDT and SeaDronesSee, boxes
through SAM3). A model trained on this pool has therefore *seen* AeroVIS's
frames, and AeroVIS stops being a held-out evaluation for it — frame for
frame, not in spirit. Clean video evaluations that remain: UAVScenes, MVUAV.
`docs/datasets.md` carries the full overlap table.

## Licences

VisDrone's mirror carries the original's academic-use terms; Kust4K is
CC BY 4.0; DroneVehicle is academic-use. The *teacher's* licence matters too:
masks made by SAM 3 inherit its use restrictions — read them once before a
deliverable depends on this pool; `facebook/sam2.1-hiera-large` is Apache-2.0
and one string away.
